In [1]:
import json
import math
from pathlib import Path
import os
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd


fpath = 'empirical_input'
fruns = 'empirical_runs.json'
INPUT = Path(os.path.join(fpath,fruns))
OUT = Path("empiricial_runs_analysis")
OUT.mkdir(exist_ok=True)
EARTH_RADIUS_KM = 6371.0088

In [2]:
# Some helpers to calculate area of query and other
def bbox_area_km2(bbox):
    """
    [north, south, east, west]
    """
    if not bbox or len(bbox) != 4:
        return np.nan

    north, south, east, west = map(float, bbox)

    return (
        EARTH_RADIUS_KM**2
        * abs(
            math.sin(math.radians(north))
            - math.sin(math.radians(south))
        )
        * math.radians(abs(east - west))
    )


def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b


def p95(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return x.quantile(0.95) if len(x) else np.nan


def cv_pct(x):
    """Coefficient of variation [%]."""
    x = pd.to_numeric(x, errors="coerce").dropna()

    if len(x) < 2 or x.mean() == 0:
        return np.nan

    return 100 * x.std(ddof=1) / x.mean()


def summary(df, by, value):
    """
    Extended metrics set.
    """
    return (
        df.groupby(by, dropna=False)[value]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
            min="min",
            max="max",
            p95=p95,
            cv_pct=cv_pct,
        )
        .reset_index()
    )


In [3]:
# load data
payload = json.loads(INPUT.read_text(encoding="utf-8"))
runs = payload.get("runs")

In [4]:
# FLATTEN DATA

run_rows = []
dispatch_rows = []
precheck_source_rows = []

for run_id, run in enumerate(runs, start=1):

    request = run.get("request", {})
    precheck = run.get("coverage_precheck", {})
    dispatch = run.get("dispatch", []) or []

    # REQUEST GEOMETRY

    bbox = request.get("bounding_box")

    if bbox and len(bbox) == 4:
        north, south, east, west = bbox
        bbox_width_deg = abs(east - west)
        bbox_height_deg = abs(north - south)
        bbox_area_deg2 = bbox_width_deg * bbox_height_deg
        area_km2 = bbox_area_km2(bbox)
    else:
        north = south = east = west = np.nan
        bbox_width_deg = bbox_height_deg = np.nan
        bbox_area_deg2 = area_km2 = np.nan

    # TIME RANGE

    time_from = pd.to_datetime(
        request.get("time_from"),
        errors="coerce"
    )

    time_to = pd.to_datetime(
        request.get("time_to"),
        errors="coerce"
    )

    if pd.notna(time_from) and pd.notna(time_to):
        # inclusive, np. Jan 1 -- Jan 7 = 7
        duration_days = (time_to - time_from).days + 1
    else:
        duration_days = np.nan

    # SOURCE TIMINGS

    source_times = [
        x["wall_seconds"]
        for x in dispatch
        if x.get("wall_seconds") is not None
    ]

    dispatch_sum = sum(source_times) if source_times else np.nan
    dispatch_max = max(source_times) if source_times else np.nan

    request_wall = run.get("request_wall_seconds", np.nan)

    # PRECHECK
    
    precheck_seconds = precheck.get("precheck_seconds", np.nan)
    candidate_sources = precheck.get("candidate_sources", np.nan)
    dispatched_sources = precheck.get("dispatched_sources", np.nan)
    requests_avoided = precheck.get("requests_avoided", np.nan)

    # OUTPUT

    cells = run.get("returned_cell_count", np.nan)
    rows = run.get("returned_row_count", np.nan)
    non_null = run.get("non_null_value_count", np.nan)

    factors = request.get("factors", []) or []

    # ONE ROW PER COMPLETE REQUEST

    run_rows.append({

        "run_id": run_id,

        "run_kind": run.get("run_kind"),
        "status": run.get("status"),
        "repeat": run.get("repeat"),

        "scenario": request.get("scenario"),
        "country": request.get("country"),

        "dimension": request.get("dimension"),
        "input_value": request.get("input_value"),

        "level": request.get("level"),

        "factor_count_requested": len(factors),
        "factors_requested": " | ".join(factors),

        "time_from": time_from,
        "time_to": time_to,
        "duration_days": duration_days,

        # geometry
        "bbox_width_deg": bbox_width_deg,
        "bbox_height_deg": bbox_height_deg,
        "bbox_area_deg2": bbox_area_deg2,
        "bbox_area_km2": area_km2,

        # total request time
        "request_wall_seconds": request_wall,

        # coverage precheck
        "precheck_seconds": precheck_seconds,
        "precheck_microseconds":
            precheck_seconds * 1e6
            if pd.notna(precheck_seconds)
            else np.nan,

        "candidate_sources": candidate_sources,
        "planned_dispatched_sources": dispatched_sources,
        "recorded_dispatch_count": len(dispatch),
        "requests_avoided": requests_avoided,

        # rejected fraction
        "avoidance_fraction":
            safe_div(requests_avoided, candidate_sources),

        "dispatch_fraction":
            safe_div(dispatched_sources, candidate_sources),

        # precheck cost
        "precheck_overhead_pct":
            100 * safe_div(
                precheck_seconds,
                request_wall
            ),

        # average check cost
        "precheck_us_per_candidate":
            1e6 * safe_div(
                precheck_seconds,
                candidate_sources
            ),

        # data_read success rate
        "dispatch_recording_gap":
            (
                dispatched_sources - len(dispatch)
                if pd.notna(dispatched_sources)
                else np.nan
            ),

        # PARALLELISM / OVERHEAD

        "dispatch_sum_seconds": dispatch_sum,
        "dispatch_max_seconds": dispatch_max,

        # slowest source
        "critical_path_share":
            safe_div(dispatch_max, request_wall),

        # ~= orchestration + merging + transformations itd.
        "orchestration_overhead_seconds":
            (
                request_wall - dispatch_max
                if pd.notna(dispatch_max)
                else np.nan
            ),

        "orchestration_overhead_pct":
            (
                100 * safe_div(
                    request_wall - dispatch_max,
                    request_wall
                )
                if pd.notna(dispatch_max)
                else np.nan
            ),


        "parallelism_factor":
            safe_div(dispatch_sum, dispatch_max),

        # RESULT SIZE

        "returned_cell_count": cells,
        "returned_row_count": rows,
        "returned_column_count":
            run.get("returned_column_count", np.nan),

        "returned_factor_count":
            run.get("returned_factor_count", np.nan),

        "non_null_value_count": non_null,

        "quality_report_count":
            run.get("quality_report_count", np.nan),

        "peak_memory_mb":
            run.get("peak_traced_memory_mb", np.nan),

        # THROUGHPUT


        "cells_per_second":
            safe_div(cells, request_wall),

        "rows_per_second":
            safe_div(rows, request_wall),

        "non_null_values_per_second":
            safe_div(non_null, request_wall),

        "seconds_per_1000_cells":
            1000 * safe_div(request_wall, cells),

        "seconds_per_million_values":
            1_000_000 * safe_div(request_wall, non_null),
    })

    # ONE ROW PER ACTUAL SOURCE REQUEST

    for d in dispatch:

        source = d.get("source")

        dispatch_rows.append({

            "run_id": run_id,

            "scenario": request.get("scenario"),
            "run_kind": run.get("run_kind"),
            "repeat": run.get("repeat"),

            "country": request.get("country"),

            "dimension": request.get("dimension"),
            "input_value": request.get("input_value"),

            "level": request.get("level"),

            "bbox_area_km2": area_km2,
            "bbox_area_deg2": bbox_area_deg2,

            "duration_days": duration_days,
            "factor_count_requested": len(factors),

            "source": source,

            "source_short":
                source.split(".")[-1]
                if source
                else None,

            "source_status": d.get("status"),
            "source_wall_seconds": d.get("wall_seconds"),

            "error": d.get("error"),
        })

    # ONE ROW PER SOURCE CONSIDERED BY PRECHECK

    for source_info in precheck.get("sources", []) or []:

        factor_overlap = source_info.get(
            "factor_overlap", []
        ) or []

        precheck_source_rows.append({

            "run_id": run_id,
            "scenario": request.get("scenario"),
            "repeat": run.get("repeat"),

            "source": source_info.get("source"),

            "source_short":
                source_info
                .get("source", "")
                .split(".")[-1],

            "dispatched":
                bool(source_info.get("dispatched")),

            "factor_overlap_count":
                len(factor_overlap),

            "spatial_overlap":
                bool(source_info.get("spatial_overlap")),

            "temporal_overlap":
                bool(source_info.get("temporal_overlap")),

            "disabled":
                source_info.get("disabled_reason")
                is not None,

            "disabled_reason":
                source_info.get("disabled_reason"),
        })


runs_df = pd.DataFrame(run_rows)
dispatch_df = pd.DataFrame(dispatch_rows)
precheck_sources_df = pd.DataFrame(precheck_source_rows)

In [5]:
runs_df

,run_id,run_kind,status,repeat,scenario,country,dimension,input_value,level,factor_count_requested,...,returned_column_count,returned_factor_count,non_null_value_count,quality_report_count,peak_memory_mb,cells_per_second,rows_per_second,non_null_values_per_second,seconds_per_1000_cells,seconds_per_million_values
0,1,coverage,success,1,germany-meteo-overlap,Germany,None,NaN,10,2,...,74.0,4.0,518.0,2.0,76.035380,0.368423,0.073685,5.452665,2714.268857,1.833965e+05
1,2,coverage,success,1,austria-meteo-overlap,Austria,None,NaN,10,2,...,88.0,4.0,616.0,2.0,8.022191,0.345205,0.058937,5.186491,2896.831388,1.928086e+05
2,3,coverage,success,1,ireland-precipitation-overlap,Ireland,None,NaN,10,1,...,25.0,1.0,175.0,1.0,1.976212,0.234587,0.065684,1.642108,4262.813132,6.089733e+05
3,4,coverage,success,1,ireland-groundwater,Ireland,None,NaN,10,1,...,1.0,1.0,90.0,1.0,81.035197,0.180460,16.241360,16.241360,5541.408000,6.157120e+04
4,5,coverage,success,1,germany-land-cover,Germany,None,NaN,10,1,...,83.0,1.0,30378.0,1.0,46.756847,2.140987,9.440979,783.601283,467.074273,1.276159e+03
5,6,coverage,success,1,germany-soil,Germany,None,NaN,10,1,...,1353.0,11.0,9471.0,1.0,3.192530,3.048752,0.173506,234.753917,328.003047,4.259780e+03
6,7,coverage,no-data,1,no-spatial-coverage,Germany,None,NaN,10,1,...,NaN,NaN,NaN,NaN,1.403971,NaN,NaN,NaN,NaN,NaN
7,8,coverage,no-data,1,no-temporal-coverage,Germany,None,NaN,10,1,...,NaN,NaN,NaN,NaN,0.012624,NaN,NaN,NaN,NaN,NaN
8,9,cross-source,success,1,cross-source-germany-meteo,Germany,None,NaN,10,2,...,24.0,2.0,744.0,1.0,22.072541,0.065034,0.168003,4.032078,15376.687842,2.480111e+05
9,10,cross-source,success,1,cross-source-austria-meteo,Austria,None,NaN,10,2,...,38.0,2.0,1178.0,1.0,2.007165,0.242818,0.396177,15.054744,4118.303105,6.642424e+04


In [6]:
dispatch_df

,run_id,scenario,run_kind,repeat,country,dimension,input_value,level,bbox_area_km2,bbox_area_deg2,duration_days,factor_count_requested,source,source_short,source_status,source_wall_seconds,error
0,1,germany-meteo-overlap,coverage,1,Germany,None,NaN,10,7781.036216,1.0,7,2,farmwise_api.adapters.API_readers.wetterdienst...,wetterdienst_dwd,success,11.928670,None
1,1,germany-meteo-overlap,coverage,1,Germany,None,NaN,10,7781.036216,1.0,7,2,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,success,82.941621,None
2,2,austria-meteo-overlap,coverage,1,Austria,None,NaN,10,8273.257240,1.0,7,2,farmwise_api.adapters.API_readers.geosphere.ge...,geosphere,success,1.942237,None
3,2,austria-meteo-overlap,coverage,1,Austria,None,NaN,10,8273.257240,1.0,7,2,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,success,116.641784,None
4,3,ireland-precipitation-overlap,coverage,1,Ireland,None,NaN,10,7354.501319,1.0,7,1,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,success,40.093074,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,58,live-s2-level-8,live-scaling,1,Germany,S2 level,8.0,8,7781.036216,1.0,7,2,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,success,23.750205,None
120,59,live-bbox-width-2.0,live-scaling,1,Germany,Bounding-box area,4.0,10,31122.959752,4.0,7,2,farmwise_api.adapters.API_readers.wetterdienst...,wetterdienst_dwd,success,28.292368,None
121,59,live-bbox-width-2.0,live-scaling,1,Germany,Bounding-box area,4.0,10,31122.959752,4.0,7,2,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,success,22.868794,None
122,60,live-factor-count-1,live-scaling,3,Germany,Factor count,1.0,10,7781.036216,1.0,7,1,farmwise_api.adapters.API_readers.wetterdienst...,wetterdienst_dwd,success,6.475186,None


In [7]:
precheck_sources_df

,run_id,scenario,repeat,source,source_short,dispatched,factor_overlap_count,spatial_overlap,temporal_overlap,disabled,disabled_reason
0,1,germany-meteo-overlap,1,farmwise_api.adapters.API_readers.geosphere.ge...,geosphere,False,2,False,True,False,None
1,1,germany-meteo-overlap,1,farmwise_api.adapters.API_readers.wetterdienst...,wetterdienst_dwd,True,2,True,True,False,None
2,1,germany-meteo-overlap,1,farmwise_api.adapters.API_readers.soilgrids.so...,soilgrids_call,False,0,True,True,False,None
3,1,germany-meteo-overlap,1,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,True,2,True,True,False,None
4,1,germany-meteo-overlap,1,farmwise_api.adapters.API_readers.cds.cds_vege...,cds_vegetation,False,0,True,True,True,The CDS sis-agroproductivity-indicators datase...
...,...,...,...,...,...,...,...,...,...,...,...
1135,60,live-factor-count-1,3,farmwise_api.adapters.API_readers.irish_meteo....,Irish MS_daily,False,0,False,True,False,None
1136,60,live-factor-count-1,3,farmwise_api.adapters.API_readers.hubeau.hubea...,hubeau_sw_quality_read,False,0,True,True,False,None
1137,60,live-factor-count-1,3,farmwise_api.adapters.API_readers.EuroCropV2.E...,EuroCropV2_read,False,0,True,True,False,None
1138,60,live-factor-count-1,3,farmwise_api.adapters.API_readers.IFSGRID.IFSG...,IFSGRID_read,False,0,True,False,True,The current upstream IFSGRID archive does not ...


In [8]:
# 1. TIME PER SOURCE
per_source = (
    dispatch_df
    .groupby(["source", "source_short"])
    .agg(
        n=("source_wall_seconds", "count"),

        mean_seconds=("source_wall_seconds", "mean"),
        median_seconds=("source_wall_seconds", "median"),
        std_seconds=("source_wall_seconds", "std"),

        p95_seconds=("source_wall_seconds", p95),

        min_seconds=("source_wall_seconds", "min"),
        max_seconds=("source_wall_seconds", "max"),

        success_rate=(
            "source_status",
            lambda x: (x == "success").mean()
        ),

        timeout_rate=(
            "source_status",
            lambda x: (x == "timeout").mean()
        ),

        failure_rate=(
            "source_status",
            lambda x: (x == "failure").mean()
        ),
    )
    .reset_index()
    .sort_values("mean_seconds", ascending=False)
)


In [9]:
per_source

,source,source_short,n,mean_seconds,median_seconds,std_seconds,p95_seconds,min_seconds,max_seconds,success_rate,timeout_rate,failure_rate
5,farmwise_api.adapters.API_readers.irish_meteo....,Irish MS_daily,2,66.314419,66.314419,0.226085,66.458298,66.154552,66.474285,0.000000,0.0,0.000000
1,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,54,35.827748,24.980731,31.599599,94.736678,15.325089,177.295302,0.907407,0.0,0.092593
6,farmwise_api.adapters.API_readers.soilgrids.so...,soilgrids_call,7,31.264901,31.640436,7.244733,39.497188,17.984509,40.300340,1.000000,0.0,0.000000
0,farmwise_api.adapters.API_readers.EuroCropV2.E...,EuroCropV2_read,4,30.373716,30.320182,0.117600,30.515466,30.304726,30.549773,0.000000,0.0,1.000000
7,farmwise_api.adapters.API_readers.wetterdienst...,wetterdienst_dwd,50,9.533213,6.650087,9.406285,33.881408,1.465972,43.057103,1.000000,0.0,0.000000
2,farmwise_api.adapters.API_readers.corine.corin...,corine_read,4,7.172038,6.892153,0.610595,7.914466,6.821293,8.082553,1.000000,0.0,0.000000
3,farmwise_api.adapters.API_readers.epa_ireland....,epa_gw,1,5.520252,5.520252,NaN,5.520252,5.520252,5.520252,1.000000,0.0,0.000000
4,farmwise_api.adapters.API_readers.geosphere.ge...,geosphere,2,1.965158,1.965158,0.032414,1.985786,1.942237,1.988078,1.000000,0.0,0.000000


In [10]:
# 2. per query times
per_query = summary(
    runs_df,
    [
        "scenario",
        "dimension",
        "input_value",
        "level",
        "country",
    ],
    "request_wall_seconds"
)
per_query

,scenario,dimension,input_value,level,country,n,mean,median,std,min,max,p95,cv_pct
0,austria-meteo-overlap,NaN,NaN,10,Austria,1,118.770087,118.770087,NaN,118.770087,118.770087,118.770087,NaN
1,cross-source-austria-meteo,NaN,NaN,10,Austria,1,78.247759,78.247759,NaN,78.247759,78.247759,78.247759,NaN
2,cross-source-germany-meteo,NaN,NaN,10,Germany,1,184.520254,184.520254,NaN,184.520254,184.520254,184.520254,NaN
3,cross-source-ireland-precipitation,NaN,NaN,10,Ireland,1,106.434735,106.434735,NaN,106.434735,106.434735,106.434735,NaN
4,cross-source-poland-meteo,NaN,NaN,10,Poland,1,55.993890,55.993890,NaN,55.993890,55.993890,55.993890,NaN
5,germany-land-cover,NaN,NaN,10,Germany,1,38.767165,38.767165,NaN,38.767165,38.767165,38.767165,NaN
6,germany-meteo-overlap,NaN,NaN,10,Germany,1,94.999410,94.999410,NaN,94.999410,94.999410,94.999410,NaN
7,germany-soil,NaN,NaN,10,Germany,1,40.344375,40.344375,NaN,40.344375,40.344375,40.344375,NaN
8,ireland-groundwater,NaN,NaN,10,Ireland,1,5.541408,5.541408,NaN,5.541408,5.541408,5.541408,NaN
9,ireland-precipitation-overlap,NaN,NaN,10,Ireland,1,106.570328,106.570328,NaN,106.570328,106.570328,106.570328,NaN


In [11]:
# 3. Level scaling test
s2 = runs_df[
    runs_df["dimension"] == "S2 level"
].copy()

per_level = summary(
    s2,
    ["level"],
    "request_wall_seconds"
)

level_extra = (
    s2.groupby("level")
    .agg(
        cells_mean=("returned_cell_count", "mean"),
        memory_mean_mb=("peak_memory_mb", "mean"),
        values_mean = ('non_null_value_count','mean'),
        values_per_second_mean=(
            "non_null_values_per_second",
            "mean"
        )
    )
    .reset_index()
)

per_level = per_level.merge(
    level_extra,
    on="level"
)
per_level

,level,n,mean,median,std,min,max,p95,cv_pct,cells_mean,memory_mean_mb,values_mean,values_per_second_mean
0,6,3,36.739043,30.095890,18.877719,22.080918,58.040320,55.245877,51.383262,4.0,21.855345,56.0,1.787231
1,8,3,29.691956,30.507073,4.007042,25.340025,33.228770,32.956600,13.495379,13.0,21.880737,182.0,6.208441
2,10,3,32.018563,31.359222,9.647439,22.717707,41.978759,40.916806,30.130769,35.0,21.852684,490.0,16.289012
3,12,3,29.641114,23.694446,10.945687,22.955985,42.272912,40.415065,36.927380,37.0,21.880625,518.0,18.893432


In [12]:
level_extra

,level,cells_mean,memory_mean_mb,values_mean,values_per_second_mean
0,6,4.0,21.855345,56.0,1.787231
1,8,13.0,21.880737,182.0,6.208441
2,10,35.0,21.852684,490.0,16.289012
3,12,37.0,21.880625,518.0,18.893432


In [13]:
# 4. BOUNDING BOX / QUERY AREA SCALING

area_runs = runs_df[
    runs_df["dimension"] == "Bounding-box area"
].copy()

per_area = summary(
    area_runs,
    [
        "input_value",
        "bbox_area_deg2",
        "bbox_area_km2",
    ],
    "request_wall_seconds"
)

area_extra = (
    area_runs
    .groupby(
        [
            "input_value",
            "bbox_area_deg2",
            "bbox_area_km2"
        ]
    )
    .agg(
        cells_mean=("returned_cell_count", "mean"),

        memory_mean_mb=(
            "peak_memory_mb",
            "mean"
        ),
                values_mean = ('non_null_value_count','mean'),
        cells_per_second_mean=(
            "cells_per_second",
            "mean"
        ),
    )
    .reset_index()
)

per_area = per_area.merge(
    area_extra,
    on=[
        "input_value",
        "bbox_area_deg2",
        "bbox_area_km2",
    ]
)

area_extra

,input_value,bbox_area_deg2,bbox_area_km2,cells_mean,memory_mean_mb,values_mean,cells_per_second_mean
0,0.0625,0.0625,486.320550,2.0,9.734332,28.0,0.088862
1,0.2500,0.2500,1945.277572,10.0,11.756326,140.0,0.359998
2,1.0000,1.0000,7781.036216,35.0,21.883234,490.0,0.984136
3,4.0000,4.0000,31122.959752,126.0,24.608763,1764.0,2.161786


In [14]:
per_area

,input_value,bbox_area_deg2,bbox_area_km2,n,mean,median,std,min,max,p95,cv_pct,cells_mean,memory_mean_mb,values_mean,cells_per_second_mean
0,0.0625,0.0625,486.320550,3,29.687159,17.981089,21.232485,16.884248,54.196141,50.574636,71.520770,2.0,9.734332,28.0,0.088862
1,0.2500,0.2500,1945.277572,3,33.799124,25.590670,19.680341,19.551735,56.254966,53.188536,58.227371,10.0,11.756326,140.0,0.359998
2,1.0000,1.0000,7781.036216,3,40.782479,32.107233,19.907492,26.684718,63.555484,60.410659,48.813837,35.0,21.883234,490.0,0.984136
3,4.0000,4.0000,31122.959752,3,61.438209,52.172808,18.416915,49.493906,82.647912,79.600402,29.976322,126.0,24.608763,1764.0,2.161786


In [15]:
# 5. NUMBER OF FACTORS SCALING


factor_runs = runs_df[
    runs_df["dimension"] == "Factor count"
].copy()

per_factor_count = summary(
    factor_runs,
    ["factor_count_requested"],
    "request_wall_seconds"
)

factor_extra = (
    factor_runs
    .groupby("factor_count_requested")
    .agg(
        returned_factors_mean=(
            "returned_factor_count",
            "mean"
        ),

        columns_mean=(
            "returned_column_count",
            "mean"
        ),

        memory_mean_mb=(
            "peak_memory_mb",
            "mean"
        ),

        values_per_second_mean=(
            "non_null_values_per_second",
            "mean"
        ),
    )
    .reset_index()
)

per_factor_count = per_factor_count.merge(
    factor_extra,
    on="factor_count_requested"
)

factor_extra

,factor_count_requested,returned_factors_mean,columns_mean,memory_mean_mb,values_per_second_mean
0,1,1.0,35.0,21.853976,8.077627
1,2,2.0,70.0,21.852655,16.672321
2,3,13.0,1423.0,21.881138,139.018128
3,4,14.0,1506.0,46.159499,106.519544


In [16]:
per_factor_count

,factor_count_requested,n,mean,median,std,min,max,p95,cv_pct,returned_factors_mean,columns_mean,memory_mean_mb,values_per_second_mean
0,1,3,32.344269,32.749760,9.646170,22.501748,41.781300,40.878146,29.823429,1.0,35.0,21.853976,8.077627
1,2,3,30.150552,32.037936,5.602386,23.848215,34.565504,34.312747,18.581373,2.0,70.0,21.852655,16.672321
2,3,3,72.085702,70.180338,6.962104,66.274651,79.802116,78.839939,9.658092,13.0,1423.0,21.881138,139.018128
3,4,3,99.493302,95.402717,9.069531,93.189539,109.887650,108.439156,9.115720,14.0,1506.0,46.159499,106.519544


In [17]:
# 6. REQUEST DURATION SCALING

duration_runs = runs_df[
    runs_df["dimension"] == "Requested days"
].copy()

per_duration = summary(
    duration_runs,
    ["duration_days"],
    "request_wall_seconds"
)

duration_extra = (
    duration_runs
    .groupby("duration_days")
    .agg(
        rows_mean=("returned_row_count", "mean"),
        cells_mean=("returned_cell_count", "mean"),
        columns_mean=("returned_column_count", "mean"),
        memory_mean_mb=("peak_memory_mb", "mean"),
        values_mean=("non_null_value_count", "mean"),
        values_per_second_mean=(
            "non_null_values_per_second",
            "mean"
        ),
    )
    .reset_index()
)

per_duration = per_duration.merge(
    duration_extra,
    on="duration_days"
)

per_duration

,duration_days,n,mean,median,std,min,max,p95,cv_pct,rows_mean,cells_mean,columns_mean,memory_mean_mb,values_mean,values_per_second_mean
0,1,3,44.408653,33.647458,19.423723,32.747286,66.831214,63.512839,43.738601,1.0,35.0,70.0,21.853584,70.0,1.755130
1,7,3,27.629473,30.140215,4.571649,22.352646,30.395559,30.370024,16.546277,7.0,35.0,70.0,21.882127,490.0,18.099822
2,30,3,36.328063,24.817783,20.071791,24.661610,59.504795,56.036094,55.251476,30.0,35.0,70.0,21.880738,2100.0,68.353536
3,90,3,98.067872,83.862584,51.058484,55.616243,154.724791,147.638570,52.064435,90.0,12.0,24.0,21.852117,2160.0,26.184756


In [18]:
# 6. SOURCE TIME PER LEVEL
source_by_level = summary(
    dispatch_df,
    ["source_short", "level"],
    "source_wall_seconds"
)

source_by_level

,source_short,level,n,mean,median,std,min,max,p95,cv_pct
0,EuroCropV2_read,10,4,30.373716,30.320182,0.117600,30.304726,30.549773,30.515466,0.387176
1,Irish MS_daily,10,2,66.314419,66.314419,0.226085,66.154552,66.474285,66.458298,0.340930
2,cds_single_levels,6,3,30.092610,23.466407,18.925606,15.371060,51.440362,48.642967,62.891209
3,cds_single_levels,8,3,22.736864,23.750205,3.935807,18.393472,26.066915,25.835244,17.310246
4,cds_single_levels,10,45,37.962356,25.072498,33.908312,15.325089,177.295302,109.901752,89.320884
5,cds_single_levels,12,3,22.634646,16.443033,11.066903,16.049323,35.411584,33.514729,48.893643
6,corine_read,10,4,7.172038,6.892153,0.610595,6.821293,8.082553,7.914466,8.513556
7,epa_gw,10,1,5.520252,5.520252,NaN,5.520252,5.520252,5.520252,NaN
8,geosphere,10,2,1.965158,1.965158,0.032414,1.942237,1.988078,1.985786,1.649445
9,soilgrids_call,10,7,31.264901,31.640436,7.244733,17.984509,40.300340,39.497188,23.172098


In [19]:
# 7. SOURCE TIME PER QUERY AREA

source_by_area = summary(
    dispatch_df[
        dispatch_df["dimension"] == "Bounding-box area"
    ],
    [
        "source_short",
        "input_value",
        "bbox_area_km2",
    ],
    "source_wall_seconds"
)
source_by_area

,source_short,input_value,bbox_area_km2,n,mean,median,std,min,max,p95,cv_pct
0,cds_single_levels,0.0625,486.320550,3,28.169493,16.474211,21.217677,15.372963,52.661304,49.042595,75.321473
1,cds_single_levels,0.2500,1945.277572,3,31.193137,23.009487,19.734366,16.867141,53.702782,50.633452,63.265090
2,cds_single_levels,1.0000,7781.036216,3,23.245531,25.005308,3.106393,19.658787,25.072498,25.065779,13.363399
3,cds_single_levels,4.0000,31122.959752,3,33.118656,24.012023,16.772959,22.868794,52.475151,49.628838,50.645048
4,wetterdienst_dwd,0.0625,486.320550,3,1.473354,1.468830,0.010408,1.465972,1.485258,1.483615,0.706413
5,wetterdienst_dwd,0.2500,1945.277572,3,2.491991,2.471044,0.075949,2.428714,2.576215,2.565698,3.047715
6,wetterdienst_dwd,1.0000,7781.036216,3,17.224793,6.794160,18.144308,6.704249,38.175971,35.037789,105.338319
7,wetterdienst_dwd,4.0000,31122.959752,3,27.274557,28.292368,2.454379,24.475013,29.056289,28.979897,8.998787


In [20]:
# 8. SOURCE TIME PER FACTOR COUNT

source_by_factor_count = summary(
    dispatch_df[
        dispatch_df["dimension"] == "Factor count"
    ],
    [
        "source_short",
        "factor_count_requested",
    ],
    "source_wall_seconds"
)
source_by_factor_count

,source_short,factor_count_requested,n,mean,median,std,min,max,p95,cv_pct
0,EuroCropV2_read,4,3,30.315030,30.319308,0.008966,30.304726,30.321056,30.320881,0.029577
1,cds_single_levels,1,3,25.492814,26.008178,9.920088,15.325089,35.145175,34.231475,38.913274
2,cds_single_levels,2,3,22.612300,24.956153,4.919974,16.958643,25.922104,25.825509,21.757955
3,cds_single_levels,3,3,26.042413,25.242768,10.848120,15.616243,37.268229,36.065683,41.655585
4,cds_single_levels,4,3,21.980505,24.627293,5.643087,15.500531,25.813690,25.695050,25.673144
5,corine_read,4,3,6.868533,6.822336,0.080921,6.821293,6.961970,6.948007,1.178136
6,soilgrids_call,3,3,32.717875,31.640436,4.465154,28.890023,37.623166,37.024893,13.647446
7,soilgrids_call,4,3,26.800113,28.895569,7.977036,17.984509,33.520262,33.057793,29.764934
8,wetterdienst_dwd,1,3,6.653596,6.475186,0.309799,6.474281,7.011320,6.957706,4.656107
9,wetterdienst_dwd,2,3,7.186912,6.654888,0.987379,6.579638,8.326211,8.159079,13.738564


In [21]:
# 9. COVERAGE PRECHECK

precheck_summary = pd.DataFrame([{

    "mean_us":
        runs_df["precheck_microseconds"].mean(),

    "median_us":
        runs_df["precheck_microseconds"].median(),

    "p95_us":
        p95(runs_df["precheck_microseconds"]),

    "max_us":
        runs_df["precheck_microseconds"].max(),

    "mean_overhead_pct":
        runs_df["precheck_overhead_pct"].mean(),

    "mean_candidates":
        runs_df["candidate_sources"].mean(),

    "mean_dispatched":
        runs_df["planned_dispatched_sources"].mean(),

    "mean_requests_avoided":
        runs_df["requests_avoided"].mean(),

    "mean_avoidance_fraction":
        runs_df["avoidance_fraction"].mean(),

    "mean_us_per_candidate":
        runs_df["precheck_us_per_candidate"].mean(),
}])
precheck_summary

,mean_us,median_us,p95_us,max_us,mean_overhead_pct,mean_candidates,mean_dispatched,mean_requests_avoided,mean_avoidance_fraction,mean_us_per_candidate
0,184.798333,183.550001,217.914998,272.499998,0.11895,19.0,2.1,16.9,0.889474,9.726228


In [22]:
# 10. WHY SOURCES ARE REJECTED

precheck_by_source = (
    precheck_sources_df
    .groupby(["source", "source_short"])
    .agg(
        considered=("run_id", "count"),

        dispatched=(
            "dispatched",
            "sum"
        ),

        disabled=(
            "disabled",
            "sum"
        ),

        no_factor_overlap=(
            "factor_overlap_count",
            lambda x: (x == 0).sum()
        ),

        no_spatial_overlap=(
            "spatial_overlap",
            lambda x: (~x).sum()
        ),

        no_temporal_overlap=(
            "temporal_overlap",
            lambda x: (~x).sum()
        ),
    )
    .reset_index()
)

precheck_by_source["dispatch_rate"] = (
    precheck_by_source["dispatched"]
    / precheck_by_source["considered"]
)
precheck_by_source

,source,source_short,considered,dispatched,disabled,no_factor_overlap,no_spatial_overlap,no_temporal_overlap,dispatch_rate
0,farmwise_api.adapters.API_readers.EuroCropV2.E...,EuroCropV2_read,60,4,0,56,0,1,0.066667
1,farmwise_api.adapters.API_readers.IFSGRID.IFSG...,IFSGRID_read,60,0,60,56,0,56,0.000000
2,farmwise_api.adapters.API_readers.UA_sw_qualit...,ukrainian_surface_water,60,0,0,60,60,0,0.000000
3,farmwise_api.adapters.API_readers.cds.cds_sing...,cds_single_levels,60,55,0,5,0,0,0.916667
4,farmwise_api.adapters.API_readers.cds.cds_vege...,cds_vegetation,60,0,60,59,0,4,0.000000
5,farmwise_api.adapters.API_readers.corine.corin...,corine_read,60,4,0,56,0,0,0.066667
6,farmwise_api.adapters.API_readers.eea.eea_read,eea_read,60,0,0,60,0,0,0.000000
7,farmwise_api.adapters.API_readers.epa_ireland....,epa_gw,60,1,0,58,57,0,0.016667
8,farmwise_api.adapters.API_readers.geosphere.ge...,geosphere,60,2,0,5,58,0,0.033333
9,farmwise_api.adapters.API_readers.gios.gios_sc...,gios_scraper,60,0,0,53,59,1,0.000000


In [23]:
# 11. REPEATABILITY / NOISE

live = runs_df[
    runs_df["run_kind"] == "live-scaling"
]

repeat_stability = (
    live.groupby(
        [
            "scenario",
            "dimension",
            "input_value",
        ]
    )
    .agg(
        repeats=("repeat", "nunique"),

        request_mean_seconds=(
            "request_wall_seconds",
            "mean"
        ),

        request_std_seconds=(
            "request_wall_seconds",
            "std"
        ),

        request_cv_pct=(
            "request_wall_seconds",
            cv_pct
        ),

        request_min_seconds=(
            "request_wall_seconds",
            "min"
        ),

        request_max_seconds=(
            "request_wall_seconds",
            "max"
        ),

        memory_mean_mb=(
            "peak_memory_mb",
            "mean"
        ),

        memory_cv_pct=(
            "peak_memory_mb",
            cv_pct
        ),
    )
    .reset_index()
    .sort_values(
        "request_cv_pct",
        ascending=False
    )
)
repeat_stability

,scenario,dimension,input_value,repeats,request_mean_seconds,request_std_seconds,request_cv_pct,request_min_seconds,request_max_seconds,memory_mean_mb,memory_cv_pct
0,live-bbox-width-0.25,Bounding-box area,0.0625,3,29.687159,21.232485,71.520770,16.884248,54.196141,9.734332,0.007639
1,live-bbox-width-0.5,Bounding-box area,0.2500,3,33.799124,19.680341,58.227371,19.551735,56.254966,11.756326,0.408010
5,live-duration-days-30,Requested days,30.0000,3,36.328063,20.071791,55.251476,24.661610,59.504795,21.880738,0.214659
7,live-duration-days-90,Requested days,90.0000,3,98.067872,51.058484,52.064435,55.616243,154.724791,21.852117,0.001958
14,live-s2-level-6,S2 level,6.0000,3,36.739043,18.877719,51.383262,22.080918,58.040320,21.855345,0.005163
2,live-bbox-width-1.0,Bounding-box area,1.0000,3,40.782479,19.907492,48.813837,26.684718,63.555484,21.883234,0.233005
4,live-duration-days-1,Requested days,1.0000,3,44.408653,19.423723,43.738601,32.747286,66.831214,21.853584,0.004684
13,live-s2-level-12,S2 level,12.0000,3,29.641114,10.945687,36.927380,22.955985,42.272912,21.880625,0.217186
12,live-s2-level-10,S2 level,10.0000,3,32.018563,9.647439,30.130769,22.717707,41.978759,21.852684,0.003758
3,live-bbox-width-2.0,Bounding-box area,4.0000,3,61.438209,18.416915,29.976322,49.493906,82.647912,24.608763,0.014387


# Cross-source

In [24]:
fpath = 'empirical_input'
fcorss = 'cross_source_observations_live.csv'
INPUT = Path(os.path.join(fpath,fcorss))
OUT = Path("observation_overview")
OUT.mkdir(exist_ok=True)
KEY = [
    "scenario",
    "timestamp",
    "cell",
    "variable",
]
df = pd.read_csv(INPUT)
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)
df["value"] = pd.to_numeric(
    df["value"],
    errors="coerce"
)

In [25]:
KEY = [
    "scenario",
    "timestamp",
    "cell",
    "variable",
]

n_sources = (
    df.groupby(KEY)["source"]
      .nunique()
      .rename("n_sources")
      .reset_index()
)

duplicated_keys = n_sources[
    n_sources["n_sources"] > 1
]

print("All unique observation keys:", len(n_sources))
print("Keys covered by >1 source:", len(duplicated_keys))
print(
    "Share with cross-source overlap:",
    f"{100 * len(duplicated_keys) / len(n_sources):.3f}%"
)

print("\nNumber of sources per observation:")
print(
    n_sources["n_sources"]
    .value_counts()
    .sort_index()
)

All unique observation keys: 2697
Keys covered by >1 source: 0
Share with cross-source overlap: 0.000%

Number of sources per observation:
1    2697
Name: n_sources, dtype: int64


In [26]:
overlap = df.merge(
    duplicated_keys[KEY],
    on=KEY,
    how="inner"
)

In [33]:
overlap

,timestamp,cell,variable,source,value,scenario


In [32]:
from itertools import combinations
pair_rows = []

for key_values, same in overlap.groupby(KEY):

    scenario, timestamp, cell, variable = key_values

    # protection against accidental duplicates within one source
    same = (
        same.groupby("source", as_index=False)
            .agg(value=("value", "mean"))
    )

    for (_, a), (_, b) in combinations(
        same.iterrows(),
        2,
    ):

        value_a = a["value"]
        value_b = b["value"]

        if pd.isna(value_a) or pd.isna(value_b):
            continue

        # alphabetical order → stable pair names
        if a["source"] <= b["source"]:
            source_a = a["source"]
            source_b = b["source"]
            va = value_a
            vb = value_b
        else:
            source_a = b["source"]
            source_b = a["source"]
            va = value_b
            vb = value_a

        diff = va - vb

        pair_rows.append({
            "scenario": scenario,
            "timestamp": timestamp,
            "cell": cell,
            "variable": variable,

            "source_a": source_a,
            "source_b": source_b,

            "source_pair":
                f"{source_a} – {source_b}",

            "value_a": va,
            "value_b": vb,

            # signed difference
            "difference": diff,

            # absolute disagreement
            "absolute_difference": abs(diff),

            # midpoint of the two values
            "pair_mean":
                (va + vb) / 2,
        })


pairs = pd.DataFrame(pair_rows)

print("\nPairwise comparisons:", len(pairs))
print("\nPairs:")
print(
    pairs["source_pair"]
    .value_counts()
)

[]

In [ ]:
def rmse(x):
    return np.sqrt(np.mean(x ** 2))


summary = (
    pairs
    .groupby(
        [
            "scenario",
            "variable",
            "source_pair",
        ]
    )
    .agg(
        n=("difference", "size"),

        # systematic difference: A - B
        mean_bias=("difference", "mean"),
        median_bias=("difference", "median"),

        # magnitude of disagreement
        mae=("absolute_difference", "mean"),
        median_absolute_difference=(
            "absolute_difference",
            "median"
        ),

        p90_absolute_difference=(
            "absolute_difference",
            lambda x: x.quantile(0.90)
        ),

        p95_absolute_difference=(
            "absolute_difference",
            lambda x: x.quantile(0.95)
        ),

        max_absolute_difference=(
            "absolute_difference",
            "max"
        ),

        rmse=(
            "difference",
            rmse
        ),
    )
    .reset_index()
)

summary

In [ ]:
# Add correlation separately
correlations = []

for keys, same in pairs.groupby(
    [
        "scenario",
        "variable",
        "source_pair",
    ]
):
    scenario, variable, source_pair = keys

    if (
        len(same) >= 2
        and same["value_a"].std() > 0
        and same["value_b"].std() > 0
    ):
        r = same[
            ["value_a", "value_b"]
        ].corr().iloc[0, 1]

    else:
        r = np.nan

    correlations.append({
        "scenario": scenario,
        "variable": variable,
        "source_pair": source_pair,
        "pearson_r": r,
    })


correlations = pd.DataFrame(correlations)
correlations

In [ ]:
summary = summary.merge(
    correlations,
    on=[
        "scenario",
        "variable",
        "source_pair",
    ],
    how="left"
)
summary

In [ ]:
variables = pairs["variable"].unique()

for variable in variables:

    sub = pairs[
        pairs["variable"] == variable
    ].copy()

    pairs_order = (
        sub.groupby("source_pair")["absolute_difference"]
           .median()
           .sort_values()
           .index
    )

    data = [
        sub.loc[
            sub["source_pair"] == pair,
            "absolute_difference"
        ].dropna()
        for pair in pairs_order
    ]
    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.boxplot(
        data,
        showfliers=False
    )
    ax.set_ylabel(
        f"Absolute difference ({variable})"
    )

    ax.set_xlabel(
        "Source pair"
    )

    ax.set_title(
        f"Cross-source disagreement: {variable}"
    )

    ax.tick_params(
        axis="x",
        rotation=30
    )

    fig.tight_layout()

    fig.savefig(
        f"cross_source_difference_{variable}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [28]:
for (
    scenario,
    variable,
    source_pair
), sub in pairs.groupby(
    [
        "scenario",
        "variable",
        "source_pair",
    ]
):

    if len(sub) < 10:
        continue

    fig, ax = plt.subplots(
        figsize=(6, 6)
    )

    ax.scatter(
        sub["value_a"],
        sub["value_b"],
        alpha=0.25,
        s=12
    )

    minimum = min(
        sub["value_a"].min(),
        sub["value_b"].min()
    )

    maximum = max(
        sub["value_a"].max(),
        sub["value_b"].max()
    )

    # perfect agreement line
    ax.plot(
        [minimum, maximum],
        [minimum, maximum],
        linestyle="--"
    )

    ax.set_xlabel(
        sub["source_a"].iloc[0]
    )

    ax.set_ylabel(
        sub["source_b"].iloc[0]
    )

    r = sub[
        ["value_a", "value_b"]
    ].corr().iloc[0, 1]

    mae = sub[
        "absolute_difference"
    ].mean()

    ax.set_title(
        f"{variable}: {source_pair}\n"
        f"{scenario} | n={len(sub):,}, "
        f"r={r:.2f}, MAE={mae:.2f}"
    )

    fig.tight_layout()

    plt.show()

KeyError: 'scenario'

In [36]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

QUALITY_DIR = INPUT.parent / payload.get(
    "quality_report_dir",
    "quality",
)
OUT_DIR = Path("quality_summary")

OUT_DIR.mkdir(exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def load_quality_report(path: Path) -> dict:
    """
    Supports:
      - JSON quality reports
      - one-row CSV quality reports

    The attached example is JSON.
    """

    # First try JSON, regardless of extension
    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            return obj

    except (json.JSONDecodeError, UnicodeDecodeError):
        pass


def to_datetime(value):
    return pd.to_datetime(
        value,
        errors="coerce",
    )


def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b


def p05(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return x.quantile(0.05) if len(x) else np.nan


def p95(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return x.quantile(0.95) if len(x) else np.nan


def cv_pct(x):
    x = pd.to_numeric(x, errors="coerce").dropna()

    if len(x) < 2 or x.mean() == 0:
        return np.nan

    return (
        100
        * x.std(ddof=1)
        / abs(x.mean())
    )


def summarise_group(group):
    """
    Summarise a set of quality reports.
    """

    result = {
        # number of reports
        "n_reports": len(group),

        # ----------------------------------------------------
        # S2 completeness
        # ----------------------------------------------------

        "S2_completeness_mean":
            group["S2_completeness"].mean(),

        "S2_completeness_median":
            group["S2_completeness"].median(),

        "S2_completeness_std":
            group["S2_completeness"].std(),

        "S2_completeness_p05":
            p05(group["S2_completeness"]),

        "S2_completeness_p95":
            p95(group["S2_completeness"]),

        "S2_completeness_min":
            group["S2_completeness"].min(),

        "S2_completeness_max":
            group["S2_completeness"].max(),

        # weighted by expected number of S2 cells
        "S2_completeness_weighted":
            safe_div(
                group["returned_s2_cells"].sum(),
                group["expected_s2_cells"].sum(),
            ),

        "expected_s2_cells_total":
            group["expected_s2_cells"].sum(),

        "returned_s2_cells_total":
            group["returned_s2_cells"].sum(),

        "missing_s2_cells_total":
            group["missing_s2_cells"].sum(),

        # ----------------------------------------------------
        # TEMPORAL COMPLETENESS
        # ----------------------------------------------------

        "missing_days_mean":
            group["missing_days"].mean(),

        "missing_days_median":
            group["missing_days"].median(),

        "missing_days_p95":
            p95(group["missing_days"]),

        "missing_days_max":
            group["missing_days"].max(),

        "reports_with_missing_days":
            (group["missing_days"] > 0).sum(),

        "reports_with_missing_days_pct":
            100 * (group["missing_days"] > 0).mean(),

        # independently derived from expected/returned
        # date boundaries
        "start_delay_days_mean":
            group["start_delay_days"].mean(),

        "end_cutshort_days_mean":
            group["end_cutshort_days"].mean(),

        "temporal_span_completeness_mean":
            group["temporal_span_completeness"].mean(),

        "temporal_span_completeness_min":
            group["temporal_span_completeness"].min(),

        # ----------------------------------------------------
        # MISSING VALUES
        # ----------------------------------------------------

        "total_missing_values_mean":
            group["total_missing_values"].mean(),

        "total_missing_values_median":
            group["total_missing_values"].median(),

        "total_missing_values_p95":
            p95(group["total_missing_values"]),

        "total_missing_values_max":
            group["total_missing_values"].max(),

        "reports_with_missing_values":
            (group["total_missing_values"] > 0).sum(),

        "reports_with_missing_values_pct":
            100
            * (group["total_missing_values"] > 0).mean(),

        "factor_missing_values_mean":
            group["factor_missing_values"].mean(),

        # ----------------------------------------------------
        # ERROR / IMPLAUSIBLE VALUES
        # ----------------------------------------------------

        "error_values_mean":
            group["error_values"].mean(),

        "error_values_median":
            group["error_values"].median(),

        "error_values_p95":
            p95(group["error_values"]),

        "error_values_max":
            group["error_values"].max(),

        "reports_with_error_values":
            (group["error_values"] > 0).sum(),

        "reports_with_error_values_pct":
            100
            * (group["error_values"] > 0).mean(),

        # ----------------------------------------------------
        # FACTOR COMPLETENESS
        # ----------------------------------------------------

        "factor_completeness_mean":
            group["factors_returned_completeness"].mean(),

        "factor_completeness_median":
            group["factors_returned_completeness"].median(),

        "factor_completeness_min":
            group["factors_returned_completeness"].min(),

        "factor_completeness_p05":
            p05(group["factors_returned_completeness"]),

        "expected_factors_total":
            group["expected_factor_count"].sum(),

        "returned_factors_total":
            group["returned_factor_count"].sum(),

        # overall weighted factor completeness
        "factor_completeness_weighted":
            safe_div(
                group["returned_factor_count"].sum(),
                group["expected_factor_count"].sum(),
            ),

        "reports_with_incomplete_factors":
            (
                group["factors_returned_completeness"] < 1
            ).sum(),

        "reports_with_incomplete_factors_pct":
            100 * (
                group["factors_returned_completeness"] < 1
            ).mean(),
    }

    return pd.Series(result)


# ============================================================
# READ ALL FILES
# ============================================================

report_rows = []
factor_rows = []

files = [
    p
    for p in QUALITY_DIR.iterdir()
    if p.is_file()
]

print(
    f"Found {len(files):,} quality report files."
)


for path in files:

    try:
        report = load_quality_report(path)

    except Exception as e:
        print(
            f"SKIPPING {path.name}: {e}"
        )
        continue

    # --------------------------------------------------------
    # BASIC VALUES
    # --------------------------------------------------------

    source = report.get("source")

    # fallback to filename if source field is absent
    if not source:
        filename = path.stem

        if "_" in filename:
            source = filename.split(
                "_",
                maxsplit=1,
            )[1]

    level = pd.to_numeric(
        report.get("S2_level"),
        errors="coerce",
    )

    expected_s2 = pd.to_numeric(
        report.get("expected_s2_cells"),
        errors="coerce",
    )

    returned_s2 = pd.to_numeric(
        report.get("returned_s2_cells"),
        errors="coerce",
    )

    # --------------------------------------------------------
    # DATES
    # --------------------------------------------------------

    expected_start = to_datetime(
        report.get("expected_start")
    )

    expected_end = to_datetime(
        report.get("expected_end")
    )

    returned_start = to_datetime(
        report.get("returned_start")
    )

    returned_end = to_datetime(
        report.get("returned_end")
    )

    if (
        pd.notna(expected_start)
        and pd.notna(expected_end)
    ):
        expected_days = (
            expected_end - expected_start
        ).days + 1
    else:
        expected_days = np.nan

    if (
        pd.notna(returned_start)
        and pd.notna(returned_end)
    ):
        returned_span_days = (
            returned_end - returned_start
        ).days + 1
    else:
        returned_span_days = np.nan

    # delay at beginning
    if (
        pd.notna(expected_start)
        and pd.notna(returned_start)
    ):
        start_delay_days = max(
            0,
            (returned_start - expected_start).days,
        )
    else:
        start_delay_days = np.nan

    # missing end
    if (
        pd.notna(expected_end)
        and pd.notna(returned_end)
    ):
        end_cutshort_days = max(
            0,
            (expected_end - returned_end).days,
        )
    else:
        end_cutshort_days = np.nan

    temporal_span_completeness = (
        safe_div(
            returned_span_days,
            expected_days,
        )
    )

    # cap at 1 in case source gives a slightly larger range
    if pd.notna(temporal_span_completeness):
        temporal_span_completeness = min(
            1.0,
            temporal_span_completeness,
        )

    # --------------------------------------------------------
    # FACTORS
    # --------------------------------------------------------

    factors_expected = (
        report.get("factors_expected", [])
        or []
    )

    factors_returned = (
        report.get("factors_returned", [])
        or []
    )

    expected_factor_count = len(
        factors_expected
    )

    returned_factor_count = len(
        factors_returned
    )

    # --------------------------------------------------------
    # REPORT-LEVEL ROW
    # --------------------------------------------------------

    report_rows.append({
        "filename": path.name,

        "request_id":
            report.get("request_id"),

        "created_at":
            to_datetime(
                report.get("created_at")
            ),

        "source":
            source,

        "api_name":
            report.get("api_name"),

        "S2_level":
            level,

        # S2
        "S2_completeness":
            pd.to_numeric(
                report.get("S2_completeness"),
                errors="coerce",
            ),

        "expected_s2_cells":
            expected_s2,

        "returned_s2_cells":
            returned_s2,

        "missing_s2_cells":
            (
                expected_s2 - returned_s2
                if (
                    pd.notna(expected_s2)
                    and pd.notna(returned_s2)
                )
                else np.nan
            ),

        # dates
        "expected_start":
            expected_start,

        "expected_end":
            expected_end,

        "returned_start":
            returned_start,

        "returned_end":
            returned_end,

        "expected_days":
            expected_days,

        "returned_span_days":
            returned_span_days,

        "start_delay_days":
            start_delay_days,

        "end_cutshort_days":
            end_cutshort_days,

        "temporal_span_completeness":
            temporal_span_completeness,

        # values supplied directly by report
        "missing_days":
            pd.to_numeric(
                report.get("missing_days"),
                errors="coerce",
            ),

        "data_delay":
            pd.to_numeric(
                report.get("data_delay"),
                errors="coerce",
            ),

        "data_cutshort":
            pd.to_numeric(
                report.get("data_cutshort"),
                errors="coerce",
            ),

        "total_missing_values":
            pd.to_numeric(
                report.get(
                    "total_missing_values"
                ),
                errors="coerce",
            ),

        "factor_missing_values":
            pd.to_numeric(
                report.get(
                    "factor_missing_values"
                ),
                errors="coerce",
            ),

        "error_values":
            pd.to_numeric(
                report.get("error_values"),
                errors="coerce",
            ),

        # factors
        "expected_factor_count":
            expected_factor_count,

        "returned_factor_count":
            returned_factor_count,

        "factors_expected":
            " | ".join(
                map(str, factors_expected)
            ),

        "factors_returned":
            " | ".join(
                map(str, factors_returned)
            ),

        "factors_returned_completeness":
            pd.to_numeric(
                report.get(
                    "factors_returned_completeness"
                ),
                errors="coerce",
            ),
    })

    # ========================================================
    # FACTOR-LEVEL QUALITY
    # ========================================================

    missing_rates = (
        report.get(
            "factor_missing_value_rates",
            {}
        )
        or {}
    )

    implausible_rates = (
        report.get(
            "implausible_value_rates",
            {}
        )
        or {}
    )

    factor_names = set(
        missing_rates
    ) | set(
        implausible_rates
    )

    for factor in factor_names:

        factor_rows.append({
            "filename":
                path.name,

            "request_id":
                report.get("request_id"),

            "source":
                source,

            "S2_level":
                level,

            "factor":
                factor,

            "missing_value_rate":
                pd.to_numeric(
                    missing_rates.get(
                        factor
                    ),
                    errors="coerce",
                ),

            "implausible_value_rate":
                pd.to_numeric(
                    implausible_rates.get(
                        factor
                    ),
                    errors="coerce",
                ),
        })


reports_df = pd.DataFrame(report_rows)
factors_df = pd.DataFrame(factor_rows)


# ============================================================
# ADD GENERAL ISSUE FLAG
# ============================================================

reports_df["has_quality_issue"] = (
    (reports_df["missing_days"] > 0)
    | (reports_df["total_missing_values"] > 0)
    | (reports_df["error_values"] > 0)
    | (
        reports_df[
            "factors_returned_completeness"
        ] < 1
    )
    | (
        reports_df[
            "S2_completeness"
        ] < 1
    )
)


# ============================================================
# SUMMARY PER SOURCE
# ============================================================

by_source = (
    reports_df
    .groupby(
        "source",
        dropna=False,
    )
    .apply(
        summarise_group,
    )
    .reset_index()
)


# ============================================================
# SUMMARY PER S2 LEVEL
# ============================================================

by_level = (
    reports_df
    .groupby(
        "S2_level",
        dropna=False,
    )
    .apply(
        summarise_group,
    )
    .reset_index()
    .sort_values("S2_level")
)


# ============================================================
# SUMMARY PER SOURCE × S2 LEVEL
# ============================================================

by_source_level = (
    reports_df
    .groupby(
        [
            "source",
            "S2_level",
        ],
        dropna=False,
    )
    .apply(
        summarise_group,
    )
    .reset_index()
    .sort_values(
        [
            "source",
            "S2_level",
        ]
    )
)


# ============================================================
# FACTOR-LEVEL SUMMARY
# ============================================================

if not factors_df.empty:

    by_factor = (
        factors_df
        .groupby(
            [
                "source",
                "factor",
            ],
            dropna=False,
        )
        .agg(
            n_reports=(
                "missing_value_rate",
                "size",
            ),

            missing_rate_mean=(
                "missing_value_rate",
                "mean",
            ),

            missing_rate_median=(
                "missing_value_rate",
                "median",
            ),

            missing_rate_p95=(
                "missing_value_rate",
                p95,
            ),

            missing_rate_max=(
                "missing_value_rate",
                "max",
            ),

            implausible_rate_mean=(
                "implausible_value_rate",
                "mean",
            ),

            implausible_rate_median=(
                "implausible_value_rate",
                "median",
            ),

            implausible_rate_p95=(
                "implausible_value_rate",
                p95,
            ),

            implausible_rate_max=(
                "implausible_value_rate",
                "max",
            ),
        )
        .reset_index()
    )

    by_factor_level = (
        factors_df
        .groupby(
            [
                "source",
                "S2_level",
                "factor",
            ],
            dropna=False,
        )
        .agg(
            n_reports=(
                "missing_value_rate",
                "size",
            ),

            missing_rate_mean=(
                "missing_value_rate",
                "mean",
            ),

            missing_rate_median=(
                "missing_value_rate",
                "median",
            ),

            missing_rate_p95=(
                "missing_value_rate",
                p95,
            ),

            implausible_rate_mean=(
                "implausible_value_rate",
                "mean",
            ),

            implausible_rate_median=(
                "implausible_value_rate",
                "median",
            ),

            implausible_rate_p95=(
                "implausible_value_rate",
                p95,
            ),
        )
        .reset_index()
    )

else:

    by_factor = pd.DataFrame()
    by_factor_level = pd.DataFrame()


# ============================================================
# ISSUE SUMMARY
#
# Useful for presentation:
# "what proportion of requests exhibit each problem?"
# ============================================================

issue_summary = (
    reports_df
    .groupby(
        "source",
        dropna=False,
    )
    .agg(
        n_reports=(
            "request_id",
            "size",
        ),

        any_quality_issue=(
            "has_quality_issue",
            "sum",
        ),

        incomplete_S2=(
            "S2_completeness",
            lambda x: (x < 1).sum(),
        ),

        missing_days=(
            "missing_days",
            lambda x: (x > 0).sum(),
        ),

        missing_values=(
            "total_missing_values",
            lambda x: (x > 0).sum(),
        ),

        error_values=(
            "error_values",
            lambda x: (x > 0).sum(),
        ),

        incomplete_factors=(
            "factors_returned_completeness",
            lambda x: (x < 1).sum(),
        ),
    )
    .reset_index()
)

for col in [
    "any_quality_issue",
    "incomplete_S2",
    "missing_days",
    "missing_values",
    "error_values",
    "incomplete_factors",
]:

    issue_summary[f"{col}_pct"] = (
        100
        * issue_summary[col]
        / issue_summary["n_reports"]
    )

Found 504 quality report files.


In [37]:
by_source

,source,n_reports,S2_completeness_mean,S2_completeness_median,S2_completeness_std,S2_completeness_p05,S2_completeness_p95,S2_completeness_min,S2_completeness_max,S2_completeness_weighted,...,reports_with_error_values_pct,factor_completeness_mean,factor_completeness_median,factor_completeness_min,factor_completeness_p05,expected_factors_total,returned_factors_total,factor_completeness_weighted,reports_with_incomplete_factors,reports_with_incomplete_factors_pct
0,adapters.API_readers.cds.cds_single_levels,127.0,0.320235,0.183824,3.269288e-01,0.013463,1.000000,0.013463,1.000000,0.145345,...,0.000000,1.0,1.0,1.0,1.0,235.0,235.0,1.0,0.0,0.0
1,adapters.API_readers.corine.corine_read,13.0,0.980747,0.977941,5.331563e-03,0.977941,0.990099,0.977941,0.990099,0.985729,...,0.000000,1.0,1.0,1.0,1.0,13.0,52.0,4.0,0.0,0.0
2,adapters.API_readers.epa_ireland.epa_gw,5.0,0.003141,0.001910,2.752537e-03,0.001910,0.006834,0.001910,0.008065,0.001916,...,80.000000,1.0,1.0,1.0,1.0,5.0,5.0,1.0,0.0,0.0
3,adapters.API_readers.geosphere.geosphere,10.0,0.215300,0.129252,1.657799e-01,0.067683,0.405941,0.067683,0.405941,0.081194,...,0.000000,1.0,1.0,1.0,1.0,16.0,16.0,1.0,0.0,0.0
4,adapters.API_readers.soilgrids.soilgrids_call,19.0,0.977941,0.977941,1.140646e-16,0.977941,0.977941,0.977941,0.977941,0.977941,...,0.000000,1.0,1.0,1.0,1.0,19.0,209.0,11.0,0.0,0.0
5,adapters.API_readers.wetterdienst.wetterdienst...,117.0,0.152561,0.088235,1.644591e-01,0.006462,0.500000,0.006462,0.596958,0.055501,...,97.435897,1.0,1.0,1.0,1.0,219.0,219.0,1.0,0.0,0.0
6,farmwise_api.adapters.API_readers.cds.cds_sing...,102.0,0.231614,0.183824,2.195595e-01,0.071429,0.928571,0.013463,1.000000,0.115943,...,0.000000,1.0,1.0,1.0,1.0,191.0,191.0,1.0,0.0,0.0
7,farmwise_api.adapters.API_readers.corine.corin...,8.0,0.977941,0.977941,1.186878e-16,0.977941,0.977941,0.977941,0.977941,0.977941,...,0.000000,1.0,1.0,1.0,1.0,8.0,32.0,4.0,0.0,0.0
8,farmwise_api.adapters.API_readers.epa_ireland....,4.0,0.008065,0.008065,0.000000e+00,0.008065,0.008065,0.008065,0.008065,0.008065,...,0.000000,1.0,1.0,1.0,1.0,4.0,4.0,1.0,0.0,0.0
9,farmwise_api.adapters.API_readers.geosphere.ge...,10.0,0.129252,0.129252,2.925695e-17,0.129252,0.129252,0.129252,0.129252,0.129252,...,0.000000,1.0,1.0,1.0,1.0,20.0,20.0,1.0,0.0,0.0


In [38]:
by_level

,S2_level,n_reports,S2_completeness_mean,S2_completeness_median,S2_completeness_std,S2_completeness_p05,S2_completeness_p95,S2_completeness_min,S2_completeness_max,S2_completeness_weighted,...,reports_with_error_values_pct,factor_completeness_mean,factor_completeness_median,factor_completeness_min,factor_completeness_p05,expected_factors_total,returned_factors_total,factor_completeness_weighted,reports_with_incomplete_factors,reports_with_incomplete_factors_pct
0,6,33.0,0.735078,0.916667,0.260804,0.363636,1.000000,0.363636,1.000000,0.790036,...,36.363636,1.0,1.0,1.0,1.0,59.0,59.0,1.00000,0.0,0.0
1,7,3.0,0.990099,0.990099,0.000000,0.990099,0.990099,0.990099,0.990099,0.990099,...,0.000000,1.0,1.0,1.0,1.0,3.0,12.0,4.00000,0.0,0.0
2,8,35.0,0.701588,0.596958,0.229299,0.405941,0.940860,0.405941,0.940860,0.804878,...,28.571429,1.0,1.0,1.0,1.0,70.0,70.0,1.00000,0.0,0.0
3,10,401.0,0.236548,0.164300,0.284581,0.053323,0.977941,0.008065,0.977941,0.190948,...,20.698254,1.0,1.0,1.0,1.0,699.0,1073.0,1.53505,0.0,0.0
4,12,32.0,0.008956,0.006462,0.004288,0.001910,0.013463,0.001910,0.013463,0.004414,...,40.625000,1.0,1.0,1.0,1.0,60.0,60.0,1.00000,0.0,0.0


In [39]:
issue_summary

,source,n_reports,any_quality_issue,incomplete_S2,missing_days,missing_values,error_values,incomplete_factors,any_quality_issue_pct,incomplete_S2_pct,missing_days_pct,missing_values_pct,error_values_pct,incomplete_factors_pct
0,adapters.API_readers.cds.cds_single_levels,127,118,118,0,0,0,0,92.913386,92.913386,0.000000,0.000000,0.000000,0.0
1,adapters.API_readers.corine.corine_read,13,13,13,0,0,0,0,100.000000,100.000000,0.000000,0.000000,0.000000,0.0
2,adapters.API_readers.epa_ireland.epa_gw,5,5,5,5,5,4,0,100.000000,100.000000,100.000000,100.000000,80.000000,0.0
3,adapters.API_readers.geosphere.geosphere,10,10,10,0,0,0,0,100.000000,100.000000,0.000000,0.000000,0.000000,0.0
4,adapters.API_readers.soilgrids.soilgrids_call,19,19,19,0,0,0,0,100.000000,100.000000,0.000000,0.000000,0.000000,0.0
5,adapters.API_readers.wetterdienst.wetterdienst...,117,117,117,4,4,114,0,100.000000,100.000000,3.418803,3.418803,97.435897,0.0
6,farmwise_api.adapters.API_readers.cds.cds_sing...,102,98,98,0,0,0,0,96.078431,96.078431,0.000000,0.000000,0.000000,0.0
7,farmwise_api.adapters.API_readers.corine.corin...,8,8,8,0,0,0,0,100.000000,100.000000,0.000000,0.000000,0.000000,0.0
8,farmwise_api.adapters.API_readers.epa_ireland....,4,4,4,4,4,0,0,100.000000,100.000000,100.000000,100.000000,0.000000,0.0
9,farmwise_api.adapters.API_readers.geosphere.ge...,10,10,10,0,0,0,0,100.000000,100.000000,0.000000,0.000000,0.000000,0.0
